# 05. Facial Expression Recognition

Extract 2308-dimensional face features that match the backend image preprocessor, train a PCA + SVM sentiment model, and save the artifacts.


In [1]:
from pathlib import Path
import sys

def _find_project_root():
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / "NeuroSense" / "webdev" / "backend").exists():
            return candidate
    raise FileNotFoundError("Could not locate the project root for this notebook.")

PROJECT_ROOT = _find_project_root()
NOTEBOOKS_DIR = PROJECT_ROOT / "NeuroSense" / "notebooks"
BACKEND_DIR = PROJECT_ROOT / "NeuroSense" / "webdev" / "backend"

for path in (NOTEBOOKS_DIR, BACKEND_DIR):
    path_str = str(path)
    if path_str not in sys.path:
        sys.path.insert(0, path_str)

from notebook_support import bootstrap_notebook

ctx = bootstrap_notebook(PROJECT_ROOT)
DATASETS_DIR = ctx["datasets_dir"]
ARTIFACTS_DIR = ctx["artifacts_dir"]
CACHE_DIR = ctx["cache_dir"]
RANDOM_STATE = ctx["random_state"]

print(f"Project root: {PROJECT_ROOT}")
print(f"Datasets directory: {DATASETS_DIR}")
print(f"Artifacts directory: {ARTIFACTS_DIR}")


Project root: /Users/devashishsingh/Desktop/human emotion recognition system
Datasets directory: /Users/devashishsingh/Desktop/human emotion recognition system/NeuroSense/datasets
Artifacts directory: /Users/devashishsingh/Desktop/human emotion recognition system/NeuroSense/artifacts


In [2]:
import joblib
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.svm import SVC

from notebook_support import collect_labeled_image_paths, extract_feature_dataset, label_counts, sample_records_by_label
from utils.emotion_utils import map_emotion_to_sentiment
from utils.preprocessors import preprocess_face_image

face_train_dir = DATASETS_DIR / "face" / "archive" / "train"
face_test_dir = DATASETS_DIR / "face" / "archive" / "test"
if not face_train_dir.exists() or not face_test_dir.exists():
    raise FileNotFoundError("Face dataset must exist at NeuroSense/datasets/face/archive/train and test.")

MAX_TRAIN_PER_SENTIMENT = 1500
MAX_TEST_PER_SENTIMENT = 500

train_records = collect_labeled_image_paths(face_train_dir, map_emotion_to_sentiment)
test_records = collect_labeled_image_paths(face_test_dir, map_emotion_to_sentiment)
train_records = sample_records_by_label(train_records, per_label=MAX_TRAIN_PER_SENTIMENT, seed=RANDOM_STATE)
test_records = sample_records_by_label(test_records, per_label=MAX_TEST_PER_SENTIMENT, seed=RANDOM_STATE)

print("Face train label counts:", label_counts(label for _, label in train_records))
print("Face test label counts:", label_counts(label for _, label in test_records))

X_train_raw, y_train_raw = extract_feature_dataset(
    train_records,
    preprocess_face_image,
    cache_path=CACHE_DIR / "face_train_sentiment.npz",
    progress_interval=200,
)
X_test_raw, y_test_raw = extract_feature_dataset(
    test_records,
    preprocess_face_image,
    cache_path=CACHE_DIR / "face_test_sentiment.npz",
    progress_interval=200,
)

print("Face feature matrix:", X_train_raw.shape, X_test_raw.shape)


Face train label counts: {'NEGATIVE': 1500, 'NEUTRAL': 1500, 'POSITIVE': 1500}
Face test label counts: {'NEGATIVE': 500, 'NEUTRAL': 500, 'POSITIVE': 500}
Face feature matrix: (4500, 2308) (1500, 2308)


In [3]:
le = LabelEncoder()
y_train = le.fit_transform(y_train_raw)
y_test = le.transform(y_test_raw)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train_raw)
X_test_sc = scaler.transform(X_test_raw)

pca = PCA(n_components=100, svd_solver="randomized", random_state=RANDOM_STATE)
X_train_pca = pca.fit_transform(X_train_sc)
X_test_pca = pca.transform(X_test_sc)

model = SVC(kernel="rbf", C=5.0, probability=True, random_state=RANDOM_STATE)
model.fit(X_train_pca, y_train)
y_pred = model.predict(X_test_pca)

print(f"Face test accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(classification_report(y_test, y_pred, target_names=le.classes_))
print("PCA explained variance:", round(float(np.sum(pca.explained_variance_ratio_)), 4))


/Users/devashishsingh/Desktop/human emotion recognition system/venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:340: RuntimeWarning: divide by zero encountered in matmul
  Q, _ = normalizer(A @ Q)
/Users/devashishsingh/Desktop/human emotion recognition system/venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:340: RuntimeWarning: overflow encountered in matmul
  Q, _ = normalizer(A @ Q)
/Users/devashishsingh/Desktop/human emotion recognition system/venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:340: RuntimeWarning: invalid value encountered in matmul
  Q, _ = normalizer(A @ Q)
/Users/devashishsingh/Desktop/human emotion recognition system/venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:341: RuntimeWarning: divide by zero encountered in matmul
  Q, _ = normalizer(A.T @ Q)
/Users/devashishsingh/Desktop/human emotion recognition system/venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:341: RuntimeWarning: overflow encountered in matmul
  Q, _ =

/Users/devashishsingh/Desktop/human emotion recognition system/venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:546: RuntimeWarning: divide by zero encountered in matmul
  U = Q @ Uhat
/Users/devashishsingh/Desktop/human emotion recognition system/venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:546: RuntimeWarning: overflow encountered in matmul
  U = Q @ Uhat
/Users/devashishsingh/Desktop/human emotion recognition system/venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:546: RuntimeWarning: invalid value encountered in matmul
  U = Q @ Uhat
/Users/devashishsingh/Desktop/human emotion recognition system/venv/lib/python3.9/site-packages/sklearn/decomposition/_base.py:153: RuntimeWarning: divide by zero encountered in matmul
  X_transformed = X @ self.components_.T
/Users/devashishsingh/Desktop/human emotion recognition system/venv/lib/python3.9/site-packages/sklearn/decomposition/_base.py:153: RuntimeWarning: overflow encountered in matmul
  X_transformed = X 

Face test accuracy: 0.5193
              precision    recall  f1-score   support

    NEGATIVE       0.50      0.49      0.49       500
     NEUTRAL       0.50      0.52      0.51       500
    POSITIVE       0.56      0.55      0.55       500

    accuracy                           0.52      1500
   macro avg       0.52      0.52      0.52      1500
weighted avg       0.52      0.52      0.52      1500

PCA explained variance: 0.8965


In [4]:
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="rocket", xticklabels=le.classes_, yticklabels=le.classes_)
plt.title("Face confusion matrix")
plt.xlabel("Predicted label")
plt.ylabel("True label")
plt.tight_layout()
plt.show()

figure, axes = plt.subplots(1, 3, figsize=(12, 4))
for axis, (sample, label) in zip(axes, zip(X_test_raw[:3], y_test_raw[:3])):
    axis.imshow(sample[:2304].reshape(48, 48), cmap="gray")
    axis.set_title(label)
    axis.axis("off")
plt.tight_layout()
plt.show()


/var/folders/y1/kwl357md29bd8ggvss23h_yc0000gn/T/ipykernel_59810/1716560726.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/var/folders/y1/kwl357md29bd8ggvss23h_yc0000gn/T/ipykernel_59810/1716560726.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [5]:
artifact_dir = ARTIFACTS_DIR / "face"
artifact_dir.mkdir(parents=True, exist_ok=True)

joblib.dump(model, artifact_dir / "face_model.pkl")
joblib.dump(scaler, artifact_dir / "face_scaler.pkl")
joblib.dump(pca, artifact_dir / "face_pca.pkl")
joblib.dump(le, artifact_dir / "face_label_encoder.pkl")

print("Saved face artifacts to:", artifact_dir)


Saved face artifacts to: /Users/devashishsingh/Desktop/human emotion recognition system/NeuroSense/artifacts/face
